In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import optuna

# =========================
# CONFIG
# =========================
N_SPLITS = 5
SEED = 42
target = "Churn"
id_col = "id"
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
# =========================
# LOAD
# =========================
y = train[target]
X = train.drop(columns=[target])
X_test = test.copy()
y = train[target]

if y.dtype == "object":
    y = y.map({"Yes": 1, "No": 0})
# =========================
# FEATURE ENGINEERING (быстрый)
# =========================
def feature_engineering(df):
    df = df.copy()

    num_cols = df.select_dtypes(include=np.number).columns

    df["num_non_null"] = df.notnull().sum(axis=1)
    df["num_missing"] = df.isnull().sum(axis=1)

    if len(num_cols) > 0:
        df["num_mean"] = df[num_cols].mean(axis=1)
        df["num_std"] = df[num_cols].std(axis=1)

    return df


X = feature_engineering(X)
X_test = feature_engineering(X_test)

# =========================
# CATEGORICAL
# =========================
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

# =========================
# ENCODING (БЫСТРЫЙ)
# =========================
for col in cat_cols:
    # Frequency
    freq = X[col].value_counts()
    X[col + "_freq"] = X[col].map(freq)
    X_test[col + "_freq"] = X_test[col].map(freq)

    # Target encoding (без KFold)
    means = y.groupby(X[col]).mean()
    X[col + "_te"] = X[col].map(means)
    X_test[col + "_te"] = X_test[col].map(means)

# fillna
global_mean = y.mean()
for col in cat_cols:
    X[col + "_te"].fillna(global_mean, inplace=True)
    X_test[col + "_te"].fillna(global_mean, inplace=True)

# category cast
for col in cat_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

# =========================
# LIGHT OPTUNA (быстро)
# =========================
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
        "num_leaves": trial.suggest_int("num_leaves", 31, 128),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.7, 0.9),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.7, 0.9),
        "seed": SEED,
        "verbose": -1,
        "num_threads": 4
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))

    for train_idx, val_idx in skf.split(X, y):
        model = lgb.train(
            params,
            lgb.Dataset(X.iloc[train_idx], y.iloc[train_idx]),
            num_boost_round=500,
            valid_sets=[lgb.Dataset(X.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[lgb.early_stopping(50)],
        )

        oof[val_idx] = model.predict(X.iloc[val_idx])

    return roc_auc_score(y, oof)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  # БЫЛО 30

best_params = study.best_params
best_params.update({
    "objective": "binary",
    "metric": "auc",
    "seed": SEED,
    "verbose": -1,
    "num_threads": 4
})

print("BEST PARAMS:", best_params)

# =========================
# TRAIN (ТОЛЬКО GBDT)
# =========================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n🚀 Fold {fold+1}")

    model = lgb.train(
        best_params,
        lgb.Dataset(X.iloc[train_idx], y.iloc[train_idx]),
        num_boost_round=1000,  # было 2000
        valid_sets=[lgb.Dataset(X.iloc[val_idx], y.iloc[val_idx])],
        callbacks=[lgb.early_stopping(100)]
    )

    oof[val_idx] = model.predict(X.iloc[val_idx])
    test_preds += model.predict(X_test) / N_SPLITS

# =========================
# SCORE
# =========================
auc = roc_auc_score(y, oof)
print(f"\n🔥 OOF ROC-AUC: {auc:.5f}")

# =========================
# SUBMISSION
# =========================
submission = pd.DataFrame({
    id_col: test[id_col],
    target: test_preds
})

submission.to_csv("submission_fast.csv", index=False)
print("✅ Submission saved!")

/tmp/ipykernel_7690/2595076858.py:72: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X[col + "_te"].fillna(global_mean, inplace=True)
/tmp/ipykernel_7690/2595076858.py:73: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[457]	valid_0's auc: 0.915341
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[422]	valid_0's auc: 0.916237
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.914795


[I 2026-03-31 18:52:06,917] Trial 0 finished with value: 0.9154508837121909 and parameters: {'learning_rate': 0.03117621451477045, 'num_leaves': 112, 'max_depth': 8, 'min_data_in_leaf': 34, 'feature_fraction': 0.7258640115526194, 'bagging_fraction': 0.774683033337086}. Best is trial 0 with value: 0.9154508837121909.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.914788
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.915816
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.914243


[I 2026-03-31 18:56:58,638] Trial 1 finished with value: 0.9149416288505562 and parameters: {'learning_rate': 0.017137409868850137, 'num_leaves': 81, 'max_depth': 7, 'min_data_in_leaf': 35, 'feature_fraction': 0.7717742020287011, 'bagging_fraction': 0.7135213900740424}. Best is trial 0 with value: 0.9154508837121909.


...

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.915385
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[497]	valid_0's auc: 0.91639
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[492]	valid_0's auc: 0.914786


[I 2026-03-31 19:25:07,198] Trial 8 finished with value: 0.9155138102708408 and parameters: {'learning_rate': 0.033924259804974634, 'num_leaves': 42, 'max_depth': 7, 'min_data_in_leaf': 27, 'feature_fraction': 0.7874829331213393, 'bagging_fraction': 0.7492097592931433}. Best is trial 2 with value: 0.9156867957992849.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[499]	valid_0's auc: 0.915174
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.916112
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.914588


[I 2026-03-31 19:28:56,667] Trial 9 finished with value: 0.9152850324089976 and parameters: {'learning_rate': 0.025029598263688037, 'num_leaves': 43, 'max_depth': 6, 'min_data_in_leaf': 75, 'feature_fraction': 0.7763212465073371, 'bagging_fraction': 0.8581554178463987}. Best is trial 2 with value: 0.9156867957992849.


BEST PARAMS: {'learning_rate': 0.04497224753197647, 'num_leaves': 119, 'max_depth': 6, 'min_data_in_leaf': 100, 'feature_fraction': 0.8654185546114455, 'bagging_fraction': 0.8765138193885513, 'objective': 'binary', 'metric': 'auc', 'seed': 42, 'verbose': -1, 'num_threads': 4}

🚀 Fold 1
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[701]	valid_0's auc: 0.915689

🚀 Fold 2
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[718]	valid_0's auc: 0.916824

🚀 Fold 3
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[695]	valid_0's auc: 0.916121

🚀 Fold 4
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[608]	valid_0's auc: 0.917206

🚀 Fold 5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[724]	valid_0's auc: 0.914433

🔥 OOF ROC-AUC: 0.91605
✅ Submission s

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n🚀 Fold {fold+1}")

    model = lgb.train(
        best_params,
        lgb.Dataset(X.iloc[train_idx], y.iloc[train_idx]),
        num_boost_round=2000,  # было 2000
        valid_sets=[lgb.Dataset(X.iloc[val_idx], y.iloc[val_idx])],
        callbacks=[lgb.early_stopping(100)]
    )

    oof[val_idx] = model.predict(X.iloc[val_idx])
    test_preds += model.predict(X_test) / N_SPLITS

# =========================
# SCORE
# =========================
auc = roc_auc_score(y, oof)
print(f"\n🔥 OOF ROC-AUC: {auc:.5f}")

# =========================
# SUBMISSION
# =========================
submission = pd.DataFrame({
    id_col: test[id_col],
    target: test_preds
})

submission.to_csv("submission_fast2.csv", index=False)
print("✅ Submission saved!")


🚀 Fold 1
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[701]	valid_0's auc: 0.915689

🚀 Fold 2
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[718]	valid_0's auc: 0.916824

🚀 Fold 3
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[695]	valid_0's auc: 0.916121

🚀 Fold 4
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[608]	valid_0's auc: 0.917206

🚀 Fold 5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[724]	valid_0's auc: 0.914433

🔥 OOF ROC-AUC: 0.91605
✅ Submission saved!
